# ERDDAP Cruise Distance Estimator

In [3]:
import math
import pandas as pd


def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth's mean radius in kilometers
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2.0) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return R * c


# Formatted ERDDAP endpoint for raw CSV output
erddap_csv_url = (
    "https://coastwatch.pfeg.noaa.gov/erddap/tabledap/fsuNoaaShipWTEPnrt.csv"
    "?longitude,latitude,time"
    "&time%3E=2026-04-30T23%3A59%3A00Z"
    "&time%3C=2026-05-22T23%3A59%3A00Z"
)

# Read data (skiprow=[1] removes ERDDAP's units row)
df = pd.read_csv(erddap_csv_url, skiprows=[1])

# Drop rows missing coordinates
df = df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

# Calculate sequential distance
total_km = 0.0
for i in range(1, len(df)):
    lat1, lon1 = df.loc[i - 1, "latitude"], df.loc[i - 1, "longitude"]
    lat2, lon2 = df.loc[i, "latitude"], df.loc[i, "longitude"]
    total_km += haversine(lat1, lon1, lat2, lon2)

total_nm = total_km * 0.539957  # Kilometers to Nautical Miles conversion

print(f"Total Data Points Processed: {len(df)}")
print(f"Total Distance Travelled: {total_km:.2f} km ({total_nm:.2f} nm)")

Total Data Points Processed: 30867
Total Distance Travelled: 8709.50 km (4702.75 nm)
